# 150 · Paper Figures & Tables (6 Models)

Six-model market impact comparison: **Historic replay**, **Heuristic price-shift**, **CST**, **LobS5 (Neural)**, **CGAN**, **RWKV**.

Toggle any model on/off via `ENABLED` list in the config cell.

Three analyses: **Square-root law (β)**, **Impact relaxation**, **Stability**.

Outputs: 7 figures (PNG) + 3 LaTeX tables.

In [1]:
import numpy as np
import pandas as pd
import re, gc, math, json
from pathlib import Path
from collections import OrderedDict
from scipy.stats import linregress
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


In [2]:
# ── Publication figure style ──────────────────────────────────────
# ACM 2-column: col width ≈ 3.33in, full width ≈ 7in
# At scale=3 export: base_px × 3 = final pixels, /300 = inches

SINGLE_W = 520   # single column base px
FULL_W   = 1080  # full width base px
IMG_SCALE = 3

_AX = dict(
    showline=True, linewidth=1.5, linecolor='black', mirror=True,
    showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.08)',
    ticks='outside', tickwidth=1, ticklen=4, tickcolor='black',
    zeroline=False,
)

def pub_layout(fig, width=SINGLE_W, height=None, legend_pos='tr', **kw):
    """Apply publication styling to a plotly figure."""
    if height is None:
        height = int(width * 0.75)
    leg = {
        'tr': dict(x=0.98, y=0.98, xanchor='right', yanchor='top'),
        'br': dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
        'tl': dict(x=0.02, y=0.98, xanchor='left',  yanchor='top'),
        'bl': dict(x=0.02, y=0.02, xanchor='left',  yanchor='bottom'),
        'tc': dict(x=0.3, y=0.98, xanchor='center', yanchor='top'),
        'none': dict(visible=False),
    }.get(legend_pos, {})
    fig.update_layout(
        width=width, height=height,
        template='plotly_white',
        font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
        title=None,
        margin=dict(l=60, r=15, t=15, b=55),
        legend=dict(**leg, bgcolor='rgba(255,255,255,0.85)',
                    bordercolor='black', borderwidth=1, font_size=11),
        **kw,
    )
    fig.update_xaxes(**_AX)
    fig.update_yaxes(**_AX)
    return fig

def save_fig(fig, name):
    """Save figure at publication DPI."""
    fig.write_image(str(FIG_DIR / name), scale=IMG_SCALE)
    print(f"  Saved: {name}")

print("Publication style loaded.")

Publication style loaded.


In [ ]:
TICK_SIZE = 100
MAX_SAMPLES = 10          # use all available
MIDPRICE_MAX = None         # no outlier filter (prices are raw LOBSTER format ~900000)
N_BOOTSTRAP = 1000

# ╔══════════════════════════════════════════════════════════════════╗
# ║  GRID — select parameter grid version                          ║
# ║  'c10x_v2': 10 (i,mb) pairs, c=10*i, 3 volumes  (30 configs)  ║
# ║  'v3':      7 (i,mb) pairs,  c=5,    3 volumes  (21 configs)  ║
# ║  'v4':      7 (i,mb) pairs,  c=10*i, 3 volumes  (21 configs)  ║
# ║  'all':     merge all grids                                    ║
# ╚══════════════════════════════════════════════════════════════════╝
GRID = 'v3'  # 'c10x_v2', 'v3', 'v4', or 'all'

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ENABLED — comment/uncomment lines to toggle models on/off     ║
# ╚══════════════════════════════════════════════════════════════════╝
ENABLED = [
    'Historic',
    'Heuristic',
    'CST',
    'LobS5',
    'CGAN',
    # 'RWKV',          # uncomment when RWKV experiments are ready
]

_ALL_SCENARIOS = OrderedDict([
    ('Historic',    {'key': 'historic_scenario',            'color': '#8F939A', 'dash': 'dash'}),
    ('Heuristic',   {'key': 'heuristic_scenario',           'color': '#2F5DA3', 'dash': 'dot'}),
    ('CST',         {'key': 'cst_scenario',                 'color': '#5B4B8A', 'dash': 'dashdot'}),
    ('LobS5',      {'key': 'aggressive_scenario',           'color': '#D09A3C', 'dash': 'solid'}),
    ('CGAN',        {'key': 'cgan_aggressive_scenario',     'color': '#7B4F9E', 'dash': 'longdash'}),
    ('RWKV',        {'key': 'rwkv_aggressive_scenario',     'color': '#D1637B', 'dash': 'longdashdot'}),
])
SCENARIOS = OrderedDict((k, v) for k, v in _ALL_SCENARIOS.items() if k in ENABLED)
print(f"GRID      : {GRID}")
print(f"ENABLED   : {list(SCENARIOS.keys())}  ({len(SCENARIOS)}/{len(_ALL_SCENARIOS)})")

# ── Grid subdirectories to scan ──
_GRID_DIRS = [GRID] if GRID != 'all' else ['c10x_v2', 'v3', 'v4']

# ── Paths (auto-detect Docker vs host) ──
_BASE = [Path("/app/output/evalsequences"),
         Path("/scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences")]
EVAL_BASE = next((p for p in _BASE if p.exists()), _BASE[-1])

_SDM = [Path("/app/lob_impact/sample_day_map.csv"),
        Path("/scratch/local/homes/80/georgenigm/LOBS5/lob_impact/sample_day_map.csv")]
SDM_PATH = next((p for p in _SDM if p.exists()), _SDM[-1])
SAMPLE_DAY_MAP = pd.read_csv(SDM_PATH)

_FIG = [Path(f"/app/pics_for_transfer_6_methods_{GRID}"),
        Path(f"/homes/80/georgenigm/LOBS5/pics_for_transfer_6_methods_{GRID}")]
FIG_DIR = next((p for p in _FIG if p.exists() or p.parent.exists()), _FIG[0])
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"EVAL_BASE : {EVAL_BASE}")
print(f"SDM       : {len(SAMPLE_DAY_MAP)} rows")
print(f"FIG_DIR   : {FIG_DIR}")

GRID      : c10x_v2
ENABLED   : ['Historic', 'Heuristic', 'CST', 'LobS5', 'CGAN']  (5/6)
EVAL_BASE : /app/output/evalsequences
SDM       : 2048 rows
FIG_DIR   : /app/pics_for_transfer_6_methods_c10x_v2


In [4]:
# ── Data I/O helpers ──────────────────────────────────────────────

def discover_v2_folders(buy_path, sell_path):
    pattern = re.compile(r'^i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)$')
    rows = []
    for p in sorted(buy_path.iterdir()):
        if not p.is_dir():
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        i, c, mb, V = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        sell_p = sell_path / p.name
        if not sell_p.exists():
            continue
        rows.append({'folder': p.name, 'i': i, 'c': c, 'mb': mb, 'V': V,
                     'Q_total': i * V, 'buy_path': p, 'sell_path': sell_p})
    return pd.DataFrame(rows)


def parse_folder_params_v2(folder_name):
    m = re.match(r'i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)', folder_name)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
    return None, None, None, None


def compute_midprice(book_array):
    return (book_array[:, 0] + book_array[:, 2]) / 2


def compute_midprice_returns(books, min_len):
    """Returns array (n_samples, min_len) of midprice - midprice[0]."""
    returns = []
    for sid, book_array in books.items():
        midprice = compute_midprice(book_array[:min_len])
        returns.append(midprice - midprice[0])
    return np.stack(returns, axis=0)


def is_midprice_outlier(book_array, max_mp):
    mp = compute_midprice(book_array)
    return np.any(mp > max_mp) or np.any(mp <= 0)


def load_aggressive_indices(data_path):
    f = data_path / 'aggressive_indices.csv'
    if not f.exists():
        return np.array([], dtype=int)
    idx = np.loadtxt(f, dtype=int)
    return np.atleast_1d(idx)


def discover_data_params(data_path, max_samples=None):
    cond_dir = data_path / "data_cond"
    pat = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    samples = []
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        m = pat.match(f.name)
        if m:
            samples.append((m.group(1), m.group(2), int(m.group(3))))
    samples.sort()
    if max_samples and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples


def load_folder_data(data_path, max_samples=None, max_midprice=None):
    samples = discover_data_params(data_path, max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    for ticker, date, sid in samples:
        cond_bp = data_path / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv"
        gen_bp  = data_path / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv"
        gen_mp  = data_path / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv"
        if not gen_bp.exists():
            continue
        cond_book = np.loadtxt(cond_bp, delimiter=',')
        gen_book  = np.loadtxt(gen_bp, delimiter=',')
        full_book = np.vstack([cond_book, gen_book])
        if max_midprice and is_midprice_outlier(full_book, max_midprice):
            continue
        gen_msg  = np.loadtxt(gen_mp, delimiter=',')
        cond_mp  = data_path / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv"
        cond_msg = np.loadtxt(cond_mp, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = full_book
        gen_msgs[key]  = np.vstack([cond_msg, gen_msg])
    return gen_books, gen_msgs, cond_lens


def load_all_v2(grid_df):
    all_data = {}
    for _, row in tqdm(grid_df.iterrows(), total=len(grid_df), desc='Loading'):
        try:
            bb, bm, bc = load_folder_data(row['buy_path'],  MAX_SAMPLES, MIDPRICE_MAX)
            sb, sm, sc = load_folder_data(row['sell_path'], MAX_SAMPLES, MIDPRICE_MAX)
            all_data[row['folder']] = {
                'buy':  {'books': bb, 'msgs': bm, 'cond_lens': bc},
                'sell': {'books': sb, 'msgs': sm, 'cond_lens': sc},
            }
        except Exception as e:
            print(f"  ERR {row['folder']}: {e}")
    return all_data


In [5]:
# ── Beta (square-root law) ────────────────────────────────────────

def extract_point_cloud(data, grid_df):
    eps = 1e-12
    rows = []
    for _, grow in grid_df.iterrows():
        folder = grow['folder']
        if folder not in data:
            continue
        d = data[folder]
        mb_val = grow['mb']
        aggr_buy  = load_aggressive_indices(grow['buy_path'])
        aggr_sell = load_aggressive_indices(grow['sell_path'])
        for direction, side, aggr_gen in [('BUY', d['buy'], aggr_buy),
                                           ('SELL', d['sell'], aggr_sell)]:
            if len(aggr_gen) == 0:
                continue
            books, msgs, conds = side['books'], side['msgs'], side['cond_lens']
            for sid in books:
                msg_arr, book_arr = msgs[sid], books[sid]
                junction = conds[sid]
                sample_id = sid[1]
                day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
                if day.empty:
                    continue
                H = float(day.iloc[0]['highest_price']) / TICK_SIZE
                L = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
                V_day = float(day.iloc[0]['execution_sum'])
                if H <= L or L <= 0 or V_day <= eps:
                    continue
                sigma = np.log(H / L) / 0.8325546
                alpha = np.log(max(sigma, eps))
                aggr_idx = junction + aggr_gen
                aggr_idx = aggr_idx[aggr_idx < len(msg_arr)]
                if len(aggr_idx) < 2:
                    continue
                sizes  = msg_arr[aggr_idx, 3].astype(float)
                prices = msg_arr[aggr_idx, 4].astype(float)
                ref = (book_arr[aggr_idx[0], 0] + book_arr[aggr_idx[0], 2]) / 2
                if ref <= 0:
                    continue
                Q_cum = np.cumsum(sizes)
                vwap  = np.cumsum(sizes * prices) / np.maximum(Q_cum, eps)
                imp   = np.abs((vwap - ref) / ref) if direction == 'BUY' else np.abs((ref - vwap) / ref)
                for a in range(len(aggr_idx)):
                    if imp[a] > eps:
                        rows.append({'x': np.log(Q_cum[a] / V_day),
                                     'y': np.log(imp[a]),
                                     'alpha': alpha,
                                     'sample_id': sample_id,
                                     'folder': folder, 'direction': direction,
                                     'mb': mb_val})
    if not rows:
        return pd.DataFrame(columns=['x', 'y', 'alpha', 'sample_id', 'folder', 'direction', 'mb'])
    return pd.DataFrame(rows)


def compute_global_beta(df):
    if df.empty:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    y_adj = df['y'].values - df['alpha'].values
    x = df['x'].values
    ok = np.isfinite(x) & np.isfinite(y_adj) & (x != 0)
    xv, yv = x[ok], y_adj[ok]
    if len(xv) < 2:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    beta = float(np.dot(xv, yv) / np.dot(xv, xv))
    ss_res = np.sum((yv - beta * xv) ** 2)
    ss_tot = np.sum(yv ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'beta': beta, 'r2': r2, 'n': int(ok.sum())}


def bootstrap_beta(df, n_boot=1000):
    if df.empty:
        return np.array([])
    pc = df[['x', 'y', 'alpha', 'sample_id']].copy()
    pc['y_adj'] = pc['y'] - pc['alpha']
    ok = np.isfinite(pc['x']) & np.isfinite(pc['y_adj']) & (pc['x'] != 0)
    pc = pc[ok]
    groups = {sid: g[['x', 'y_adj']].values for sid, g in pc.groupby('sample_id')}
    ids = np.array(list(groups.keys()))
    n = len(ids)
    if n == 0:
        return np.array([])
    rng = np.random.RandomState(42)
    betas = np.zeros(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(ids, size=n, replace=True)
        pool = np.vstack([groups[s] for s in chosen])
        x, y = pool[:, 0], pool[:, 1]
        betas[b] = np.dot(x, y) / np.dot(x, x)
    return betas

In [6]:
# ── Master curves, relaxation, gamma ──────────────────────────────

def compute_master_curve(buy_data, sell_data, folder, aggr_gen,
                         u_max=11.0, n_pts=500):
    """Sigma-normalized volume-time impact curve for one folder."""
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            sample_id = sid[1]
            day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
            if day.empty:
                continue
            H = float(day.iloc[0]['highest_price']) / TICK_SIZE
            Lp = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
            if H <= Lp or Lp <= 0:
                continue
            sigma = np.log(H / Lp) / 0.8325546
            if sigma <= 0:
                continue
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            if ref <= 0:
                continue
            raw = (mid[s_abs:min_len] - ref) / (ref * sigma)
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_raw_curve(buy_data, sell_data, folder, aggr_gen,
                      u_max=11.0, n_pts=500):
    """Raw (absolute) volume-time impact curve — matches NB 120 logic."""
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            raw = mid[s_abs:min_len] - ref
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_combined_impact(buy_data, sell_data, folder):
    """Absolute (buy-sell)/2 midprice returns — matches NB 120 logic."""
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    buy_returns = compute_midprice_returns(bb, min_len)
    sell_returns = compute_midprice_returns(sb, min_len)
    combined = (buy_returns.mean(axis=0) - sell_returns.mean(axis=0)) / 2
    junction = list(buy_data['cond_lens'].values())[0]
    post = combined[junction:]
    if len(post) == 0:
        return None
    pk_idx = junction + np.argmax(post)
    return {'mean': combined, 'junction': junction,
            'peak': float(combined[pk_idx]), 'final': float(combined[-1]),
            'peak_idx': pk_idx,
            'n_buy': buy_returns.shape[0], 'n_sell': sell_returns.shape[0],
            'min_len': min_len}

In [7]:
# ── Stability (3-method vote) ─────────────────────────────────────

TAIL_FRAC   = 0.20
WINDOW_FRAC = 0.15
SLOPE_THRESH = 0.05
TWIN_THRESH  = 0.03
CONV_THRESH  = 95.0


def stability_for_folder(stats, row):
    """3-method stability test for one folder."""
    if stats is None:
        return {'folder': row['folder'], 'stabilized': False, 'votes': 0}
    mean_curve = stats['mean']
    junction   = stats['junction']
    post = mean_curve[junction:]
    pk_local = np.argmax(post)
    peak_val = post[pk_local]
    post_peak = post[pk_local:]
    final_val = post_peak[-1]
    n = len(post_peak)
    if n < 3 or abs(peak_val) < 1e-12:
        return {'folder': row['folder'], 'stabilized': False, 'votes': 0}
    # M1: trailing slope
    tl = max(int(n * TAIL_FRAC), 3)
    tail = post_peak[-tl:]
    sl = linregress(np.arange(tl, dtype=float), tail)
    slope_ok = abs(sl.slope * tl / peak_val) < SLOPE_THRESH
    # M2: two-window
    w = max(int(n * WINDOW_FRAC), 3)
    if 2 * w <= n:
        twin_ok = abs((np.mean(post_peak[-w:]) - np.mean(post_peak[-2*w:-w])) / peak_val) < TWIN_THRESH
    else:
        twin_ok = False
    # M3: exponential fit
    exp_ok = False
    try:
        t = np.arange(n, dtype=float)
        def ef(t, A, tau, C):
            return A * np.exp(-t / tau) + C
        popt, _ = curve_fit(ef, t, post_peak,
                            p0=[float(peak_val - final_val), n / 3.0, float(final_val)],
                            maxfev=10000)
        conv = (1.0 - np.exp(-n / popt[1])) * 100 if popt[1] > 0 else 100.0
        exp_ok = conv > CONV_THRESH
    except Exception:
        pass
    votes = slope_ok + twin_ok + exp_ok
    return {'folder': row['folder'], 'stabilized': votes >= 2, 'votes': int(votes)}


In [8]:
# ── Load & process all enabled scenarios ──────────────────────────

R = OrderedDict()  # results dict

for label, cfg in SCENARIOS.items():
    # ── Discover folders across selected grid subdirectories ──
    grid_frames = []
    for gdir in _GRID_DIRS:
        buy_p  = EVAL_BASE / cfg['key'] / gdir / 'context_500_buy'
        sell_p = EVAL_BASE / cfg['key'] / gdir / 'context_500_sell'
        if buy_p.exists() and sell_p.exists():
            gf = discover_v2_folders(buy_p, sell_p)
            if not gf.empty:
                gf['grid_version'] = gdir
                grid_frames.append(gf)
    if not grid_frames:
        print(f"SKIP {label}: no folders found in {_GRID_DIRS}")
        continue
    grid = pd.concat(grid_frames, ignore_index=True)
    print(f"\n{'='*60}\n  {label}: {len(grid)} configs (grids: {grid['grid_version'].unique().tolist()})")
    data = load_all_v2(grid)
    print(f"  Loaded {len(data)} folders, keys sample: {list(data.keys())[:3]}")

    # Debug: check one folder's structure
    if data:
        _f0 = next(iter(data))
        _d0 = data[_f0]
        print(f"  Folder '{_f0}':")
        print(f"    buy  books={len(_d0['buy']['books'])}  msgs={len(_d0['buy']['msgs'])}")
        print(f"    sell books={len(_d0['sell']['books'])} msgs={len(_d0['sell']['msgs'])}")
        _aggr = load_aggressive_indices(grid.iloc[0]['buy_path'])
        print(f"    aggressive_indices: {_aggr} (len={len(_aggr)})")

    # ── Beta (exclude mb=20, like NB 120 pc_no20) ──
    pc_all = extract_point_cloud(data, grid)
    pc = pc_all[pc_all['mb'] != 20] if len(pc_all) > 0 else pc_all
    print(f"  Point cloud: {len(pc_all)} total, {len(pc)} after mb!=20 filter")
    bstat = compute_global_beta(pc)
    bbetas = bootstrap_beta(pc, N_BOOTSTRAP)

    # ── Sigma-normalized master curves (for Figure 1, 2) ──
    curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        c = compute_master_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if c is not None:
            curves[f] = c

    # ── Raw curves (for relaxation ratio — matches NB 120) ──
    raw_curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        c = compute_raw_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if c is not None:
            raw_curves[f] = c

    # ── Combined impact (absolute returns) → stability + gamma ──
    stab_rows, metrics_rows = [], []
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        st = compute_combined_impact(data[f]['buy'], data[f]['sell'], f)
        stab_rows.append(stability_for_folder(st, row))
        if st is not None:
            metrics_rows.append({'folder': f, 'i': row['i'], 'mb': row['mb'],
                                 'V': row['V'], 'peak': st['peak']})

    stab_df = pd.DataFrame(stab_rows)
    metrics_df = pd.DataFrame(metrics_rows) if metrics_rows else pd.DataFrame()

    # ── Gamma (V-scaling) ──
    gamma_rows = []
    if not metrics_df.empty:
        for (iv, mbv), grp in metrics_df.groupby(['i', 'mb']):
            grp = grp.sort_values('V')
            if len(grp) < 3:
                continue
            Vs = grp['V'].values.astype(float)
            pks = grp['peak'].values
            if np.any(pks <= 0) or np.any(Vs <= 0):
                continue
            sl = linregress(np.log(Vs), np.log(pks))
            gamma_rows.append({'i': iv, 'mb': mbv, 'gamma': sl.slope,
                               'r2': sl.rvalue**2, 'se': sl.stderr})
    gamma_df = pd.DataFrame(gamma_rows) if gamma_rows else pd.DataFrame()

    # ── Relaxation ratios (from RAW curves, like NB 120) ──
    relax_rows = []
    for f, cv in raw_curves.items():
        u, m = cv['u_grid'], cv['combined_mean']
        I_peak = float(np.interp(1.0, u, m))
        if abs(I_peak) < 1e-12:
            continue
        I_final = float(m[-1])
        relax_rows.append({'folder': f, 'I_peak': I_peak, 'I_final': I_final,
                           'ratio': I_final / I_peak,
                           'mb': cv['mb'], 'V': cv['V']})
    relax_df = pd.DataFrame(relax_rows) if relax_rows else pd.DataFrame()

    R[label] = {
        'grid': grid, 'pc': pc, 'beta': bstat, 'boot': bbetas,
        'curves': curves, 'raw_curves': raw_curves, 'relax_df': relax_df,
        'stab_df': stab_df, 'gamma_df': gamma_df,
    }
    stable_frac = stab_df['stabilized'].mean() if not stab_df.empty else 0
    relax_med = relax_df['ratio'].median() if not relax_df.empty else np.nan
    print(f"  β={bstat['beta']:.4f}  R²={bstat['r2']:.4f}  n={bstat['n']:,}")
    print(f"  Curves: {len(curves)} sigma-norm, {len(raw_curves)} raw")
    print(f"  Relax median: {relax_med:.3f}, Stable: {stable_frac:.0%}, Gamma: {len(gamma_rows)}")

    del data
    gc.collect()

print(f"\n{'='*60}\nLoaded {len(R)} scenarios: {list(R.keys())}")


  Historic: 30 configs (grids: ['c10x_v2'])


Loading:   0%|          | 0/30 [00:00<?, ?it/s]

  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=100  msgs=100
    sell books=100 msgs=100
    aggressive_indices: [20] (len=1)
  Point cloud: 19162 total, 18025 after mb!=20 filter
  β=0.5516  R²=0.9490  n=18,025
  Curves: 27 sigma-norm, 27 raw
  Relax median: 0.066, Stable: 0%, Gamma: 10

  Heuristic: 30 configs (grids: ['c10x_v2'])


Loading:   0%|          | 0/30 [00:00<?, ?it/s]

  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=100  msgs=100
    sell books=100 msgs=100
    aggressive_indices: [20] (len=1)
  Point cloud: 19137 total, 18008 after mb!=20 filter
  β=0.5505  R²=0.9500  n=18,008
  Curves: 27 sigma-norm, 27 raw
  Relax median: 1.399, Stable: 0%, Gamma: 10

  CST: 30 configs (grids: ['c10x_v2'])


...

  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=100  msgs=100
    sell books=100 msgs=100
    aggressive_indices: [20] (len=1)
  Point cloud: 18874 total, 17751 after mb!=20 filter
  β=0.5644  R²=0.9567  n=17,751
  Curves: 27 sigma-norm, 27 raw
  Relax median: 0.811, Stable: 0%, Gamma: 10

  LobS5: 30 configs (grids: ['c10x_v2'])


Loading:   0%|          | 0/30 [00:00<?, ?it/s]

  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=100  msgs=100
    sell books=100 msgs=100
    aggressive_indices: [20] (len=1)
  Point cloud: 19081 total, 17957 after mb!=20 filter
  β=0.5614  R²=0.9503  n=17,957
  Curves: 27 sigma-norm, 27 raw
  Relax median: 0.706, Stable: 0%, Gamma: 10

  CGAN: 30 configs (grids: ['c10x_v2'])


Loading:   0%|          | 0/30 [00:00<?, ?it/s]

  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=100  msgs=100
    sell books=100 msgs=100
    aggressive_indices: [20] (len=1)
  Point cloud: 19798 total, 18598 after mb!=20 filter
  β=0.5861  R²=0.9875  n=18,598
  Curves: 27 sigma-norm, 27 raw
  Relax median: 0.969, Stable: 0%, Gamma: 10

Loaded 5 scenarios: ['Historic', 'Heuristic', 'CST', 'LobS5', 'CGAN']


---
## 1. Square-Root Law (β)

**Theory**: $\Delta p / \sigma \sim (Q/V)^\beta$ with $\beta = 0.5$ (Kyle, 1985; Tóth et al., 2011).


In [9]:
# ── Table 1: Global Beta Comparison ──

rows = []
for label, r in R.items():
    b = r['beta']
    ci = np.percentile(r['boot'], [2.5, 97.5]) if len(r['boot']) > 0 else [np.nan, np.nan]
    rows.append({'Model': label, 'β': f"{b['beta']:.3f}",
                 'R²': f"{b['r2']:.3f}", 'N': f"{b['n']:,}",
                 '95% CI': f"[{ci[0]:.3f}, {ci[1]:.3f}]"})
table1 = pd.DataFrame(rows)
print("\n── Table 1: Global Beta Comparison ──")
print(table1.to_string(index=False))

# LaTeX
print("\n── LaTeX ──")
print("\\begin{tabular}{lcccc}")
print("\\toprule")
print("Model & $\\beta$ & $R^2$ & $N$ & 95\\% CI \\\\")
print("\\midrule")
for _, r in table1.iterrows():
    print(f"{r['Model']} & {r['β']} & {r['R²']} & {r['N']} & {r['95% CI']} \\\\")
print("\\bottomrule")
print("\\end{tabular}")



── Table 1: Global Beta Comparison ──
    Model     β    R²      N         95% CI
 Historic 0.552 0.949 18,025 [0.540, 0.563]
Heuristic 0.551 0.950 18,008 [0.540, 0.561]
      CST 0.564 0.957 17,751 [0.557, 0.571]
    LobS5 0.561 0.950 17,957 [0.552, 0.570]
     CGAN 0.586 0.987 18,598 [0.579, 0.593]

── LaTeX ──
\begin{tabular}{lcccc}
\toprule
Model & $\beta$ & $R^2$ & $N$ & 95\% CI \\
\midrule
Historic & 0.552 & 0.949 & 18,025 & [0.540, 0.563] \\
Heuristic & 0.551 & 0.950 & 18,008 & [0.540, 0.561] \\
CST & 0.564 & 0.957 & 17,751 & [0.557, 0.571] \\
LobS5 & 0.561 & 0.950 & 17,957 & [0.552, 0.570] \\
CGAN & 0.586 & 0.987 & 18,598 & [0.579, 0.593] \\
\bottomrule
\end{tabular}


In [10]:
# ── Figure 3: Beta Regression Lines (single column) ──

fig = go.Figure()
x_range = np.array([-16, -4])

for label, r in R.items():
    pc = r['pc']
    if pc.empty:
        continue
    color = SCENARIOS[label]['color']
    dash  = SCENARIOS[label]['dash']
    beta  = r['beta']['beta']

    # scatter (subsample)
    n_show = min(5000, len(pc))
    idx = np.random.RandomState(42).choice(len(pc), n_show, replace=False)
    sub = pc.iloc[idx]
    fig.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'] - sub['alpha'], mode='markers',
        marker=dict(size=2.5, color=color, opacity=0.12),
        name=label, showlegend=False))
    # regression line
    fig.add_trace(go.Scatter(
        x=x_range, y=beta * x_range, mode='lines',
        line=dict(color=color, width=2.5, dash=dash),
        name=f"{label} (\u03b2={beta:.3f})"))

# theory line
fig.add_trace(go.Scatter(
    x=x_range, y=0.5 * x_range, mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    name='Theory (\u03b2=0.5)'))

pub_layout(fig, width=SINGLE_W, height=420, legend_pos='br')
fig.update_xaxes(title_text='ln(Q / V)')
fig.update_yaxes(title_text='ln(I / \u03c3)')
save_fig(fig, '3. Beta Regression Lines.png')
fig.show()

  Saved: 3. Beta Regression Lines.png


In [11]:
# ── Figure 4: Bootstrap Beta Distributions (single column) ──

fig = go.Figure()
for label, r in R.items():
    bb = r['boot']
    if len(bb) == 0:
        continue
    fig.add_trace(go.Histogram(
        x=bb, nbinsx=50, name=label, opacity=0.55,
        marker_color=SCENARIOS[label]['color'],
        marker_line_color='black', marker_line_width=0.5))

fig.add_vline(x=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='\u03b2 = 0.5', annotation_font_size=11,
              annotation_position='top left')

pub_layout(fig, width=SINGLE_W, height=360, legend_pos='tc',
           barmode='overlay')
fig.update_xaxes(title_text='\u03b2 (bootstrap)')
fig.update_yaxes(title_text='Count')
save_fig(fig, '4. Bootstrap Beta Distributions.png')
fig.show()

  Saved: 4. Bootstrap Beta Distributions.png


---
## 2. Master Curves & Impact Relaxation

**Theory**: after rescaling by $Q^\beta \sigma$, individual impact curves should collapse
onto a universal master curve $\mathcal{F}(u)$ with relaxation ratio
$\mathcal{F}(\infty)/\mathcal{F}(1) \approx 2/3$ (Bouchaud et al., 2004).


In [ ]:
# ── Figure 1: Master Curves (full width, dynamic grid) ──

# Adaptive u limit (same as Figure 2)
_u_all = []
for label, r in R.items():
    for f, c in r['curves'].items():
        _u_all.append(c['u_grid'][-1])
U_MASTER = min(3.0, np.percentile(_u_all, 10)) if _u_all else 3.0
print(f"  Master curves plot limit: u <= {U_MASTER:.2f}")

n_scn = len(R)
n_cols = 2
n_rows = math.ceil(n_scn / n_cols)
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'<b>{l}</b>' for l in R.keys()],
    horizontal_spacing=0.10, vertical_spacing=0.13)

palette = px.colors.qualitative.D3 + px.colors.qualitative.Set2

for idx, (label, r) in enumerate(R.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    sorted_f = sorted(r['curves'].keys(),
        key=lambda f: (parse_folder_params_v2(f)[2], parse_folder_params_v2(f)[0]))
    for fi, folder in enumerate(sorted_f):
        c = r['curves'][folder]
        u, m = c['u_grid'], c['combined_mean']
        mask = u <= U_MASTER
        fig.add_trace(go.Scatter(
            x=u[mask], y=m[mask], mode='lines',
            line=dict(color=palette[fi % len(palette)], width=1.2),
            showlegend=False), row=row, col=col)
    fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)',
                  line_width=1, row=row, col=col)

fig.update_layout(
    width=FULL_W, height=int(FULL_W * 0.36 * n_rows),
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    title=None,
    margin=dict(l=55, r=15, t=35, b=50),
)
fig.update_xaxes(**_AX, title_text='u = n / L', title_font_size=12, tickfont_size=10)
fig.update_yaxes(**_AX, title_text='I<sub>norm</sub>(u)', title_font_size=12, tickfont_size=10)
fig.update_annotations(font_size=13)
save_fig(fig, '1. Master Curves.png')
fig.show()

In [ ]:
# ── Figure 2: Average Master Curve (single column) ──

fig = go.Figure()

# Adaptive u_max: use the smallest max-u across all scenario curves
u_max_all = []
for label, r in R.items():
    for f, c in r['curves'].items():
        u_max_all.append(c['u_grid'][-1])
U_PLOT = min(3.0, np.percentile(u_max_all, 10)) if u_max_all else 3.0
u_common = np.linspace(0, U_PLOT, 300)
print(f"  Average master curve u_max = {U_PLOT:.2f}  (10th percentile of {len(u_max_all)} curves)")

for label, r in R.items():
    interps = []
    for f, c in r['curves'].items():
        if c['u_grid'][-1] >= U_PLOT:
            interps.append(np.interp(u_common, c['u_grid'], c['combined_mean']))
    if not interps:
        print(f"  WARNING: {label} — no curves reach u={U_PLOT:.2f}, skipping")
        continue
    avg = np.mean(interps, axis=0)
    std = np.std(interps, axis=0)
    color = SCENARIOS[label]['color']

    # hex -> rgba for fill band
    rc, gc, bc = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fill_rgba = f'rgba({rc},{gc},{bc},0.12)'

    fig.add_trace(go.Scatter(
        x=np.concatenate([u_common, u_common[::-1]]),
        y=np.concatenate([avg + std, (avg - std)[::-1]]),
        fill='toself', fillcolor=fill_rgba,
        line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(
        x=u_common, y=avg, mode='lines',
        line=dict(color=color, width=2.5), name=label))

fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=SINGLE_W, height=380, legend_pos='tr')
fig.update_xaxes(title_text='u = n / L')
fig.update_yaxes(title_text='I<sub>norm</sub>(u)')
save_fig(fig, '2. Average Master Curve.png')
fig.show()

In [14]:
# ── Table 2: Relaxation Ratio ──

rows = []
for label, r in R.items():
    rdf = r['relax_df']
    if rdf.empty:
        continue
    med   = rdf['ratio'].median()
    mean  = rdf['ratio'].mean()
    std   = rdf['ratio'].std()
    cv    = std / abs(mean) if abs(mean) > 1e-12 else np.nan
    rows.append({'Model': label,
                 'Median ratio': f"{med:.3f}",
                 'Mean \u00b1 std': f"{mean:.3f} \u00b1 {std:.3f}",
                 'CV': f"{cv:.3f}",
                 '\u0394 from 2/3': f"{abs(med - 2/3):.3f}"})
table2 = pd.DataFrame(rows)
print("\n\u2500\u2500 Table 2: Relaxation Ratio (I_final / I_peak) \u2500\u2500")
print("  Bouchaud theoretical: 2/3 \u2248 0.667")
print(table2.to_string(index=False))

# LaTeX
print("\n\u2500\u2500 LaTeX \u2500\u2500")
print("\\begin{tabular}{lcccr}")
print("\\toprule")
print("Model & Median & Mean $\\pm$ std & CV & $|\\Delta|$ from $\\frac{2}{3}$ \\\\")
print("\\midrule")
for _, r in table2.iterrows():
    print(f"{r['Model']} & {r['Median ratio']} & {r['Mean \u00b1 std']} & {r['CV']} & {r['\u0394 from 2/3']} \\\\")
print("\\bottomrule")
print("\\end{tabular}")

# ── Figure 5: Relaxation Ratio box plot (single column) ──
fig = go.Figure()
for label, r in R.items():
    rdf = r['relax_df']
    if rdf.empty:
        continue
    fig.add_trace(go.Box(
        y=rdf['ratio'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=2/3, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='2/3', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=SINGLE_W, height=380, legend_pos='none')
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='I<sub>final</sub> / I<sub>peak</sub>')
save_fig(fig, '5. Relaxation Ratio.png')
fig.show()


── Table 2: Relaxation Ratio (I_final / I_peak) ──
  Bouchaud theoretical: 2/3 ≈ 0.667
    Model Median ratio      Mean ± std    CV Δ from 2/3
 Historic        0.066   0.072 ± 0.031 0.435      0.600
Heuristic        1.399   1.409 ± 0.075 0.053      0.733
      CST        0.811   0.817 ± 0.127 0.156      0.145
    LobS5        0.706   0.664 ± 0.398 0.600      0.040
     CGAN        0.969 -3.407 ± 31.477 9.240      0.302

── LaTeX ──
\begin{tabular}{lcccr}
\toprule
Model & Median & Mean $\pm$ std & CV & $|\Delta|$ from $\frac{2}{3}$ \\
\midrule
Historic & 0.066 & 0.072 ± 0.031 & 0.435 & 0.600 \\
Heuristic & 1.399 & 1.409 ± 0.075 & 0.053 & 0.733 \\
CST & 0.811 & 0.817 ± 0.127 & 0.156 & 0.145 \\
LobS5 & 0.706 & 0.664 ± 0.398 & 0.600 & 0.040 \\
CGAN & 0.969 & -3.407 ± 31.477 & 9.240 & 0.302 \\
\bottomrule
\end{tabular}
  Saved: 5. Relaxation Ratio.png


---
## 3. Stability

A configuration is **stable** if ≥2 of 3 methods agree: trailing slope, two-window mean,
exponential convergence. Fraction stable = share of 30 configs that stabilize.


In [15]:
# ── Table 3: Fraction Stable ──

rows = []
for label, r in R.items():
    sdf = r['stab_df']
    frac = sdf['stabilized'].mean() if not sdf.empty else 0.0
    n_stable = int(sdf['stabilized'].sum()) if not sdf.empty else 0
    n_total  = len(sdf) if not sdf.empty else 0
    rows.append({'Model': label,
                 'Stable': n_stable, 'Total': n_total,
                 'Fraction': f"{frac:.0%}"})
table3 = pd.DataFrame(rows)
print("\n\u2500\u2500 Table 3: Fraction Stable (2+/3 votes) \u2500\u2500")
print(table3.to_string(index=False))

# LaTeX
print("\n\u2500\u2500 LaTeX \u2500\u2500")
print("\\begin{tabular}{lccc}")
print("\\toprule")
print("Model & Stable / Total & Fraction \\\\")
print("\\midrule")
for _, r in table3.iterrows():
    print(f"{r['Model']} & {r['Stable']}/{r['Total']} & {r['Fraction']} \\\\")
print("\\bottomrule")
print("\\end{tabular}")

# ── Figure 6: Fraction Stable bar chart (single column) ──
labels = [r['Model'] for _, r in table3.iterrows()]
fracs  = [float(r['Fraction'].strip('%')) / 100 for _, r in table3.iterrows()]
colors = [SCENARIOS[l]['color'] for l in labels]

fig = go.Figure(go.Bar(
    x=labels, y=fracs, marker_color=colors, width=0.55,
    marker_line_color='black', marker_line_width=1))

fig.add_hline(y=0.5, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=SINGLE_W, height=360, legend_pos='none',
           yaxis_range=[0, 1.05])
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='Fraction stable (\u22652/3 votes)')
save_fig(fig, '6. Fraction Stable.png')
fig.show()


── Table 3: Fraction Stable (2+/3 votes) ──
    Model  Stable  Total Fraction
 Historic       0     30       0%
Heuristic       0     30       0%
      CST       0     30       0%
    LobS5       0     30       0%
     CGAN       0     30       0%

── LaTeX ──
\begin{tabular}{lccc}
\toprule
Model & Stable / Total & Fraction \\
\midrule
Historic & 0/30 & 0% \\
Heuristic & 0/30 & 0% \\
CST & 0/30 & 0% \\
LobS5 & 0/30 & 0% \\
CGAN & 0/30 & 0% \\
\bottomrule
\end{tabular}
  Saved: 6. Fraction Stable.png


---
## 4. Supplementary: Volume Scaling (γ)

$\text{Peak impact} \propto V^\gamma$. Square-root law predicts $\gamma = \beta \approx 0.5$.


In [16]:
# ── Figure 99: Gamma Distribution (single column) ──

fig = go.Figure()
for label, r in R.items():
    gdf = r['gamma_df']
    if gdf.empty:
        continue
    fig.add_trace(go.Box(
        y=gdf['gamma'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='\u03b3 = 0.5', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=SINGLE_W, height=380, legend_pos='none')
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='\u03b3 (volume scaling exponent)')
save_fig(fig, '99. Gamma Distribution.png')
fig.show()

# Summary
for label, r in R.items():
    gdf = r['gamma_df']
    if not gdf.empty:
        print(f"{label:15s}  \u03b3 = {gdf['gamma'].mean():.3f} \u00b1 {gdf['gamma'].std():.3f}  (n={len(gdf)})")

  Saved: 99. Gamma Distribution.png


Historic         γ = 0.910 ± 0.092  (n=10)
Heuristic        γ = 0.893 ± 0.083  (n=10)
CST              γ = 0.712 ± 0.121  (n=10)
LobS5            γ = 0.779 ± 0.154  (n=10)
CGAN             γ = 0.560 ± 0.297  (n=10)


In [17]:
# ── Summary ──

print("=" * 70)
print(f"{'Model':15s}  {'β':>6s}  {'Relax':>6s}  {'Stable':>7s}  {'γ':>6s}")
print("-" * 70)
for label, r in R.items():
    beta = r['beta']['beta']
    relax_med = r['relax_df']['ratio'].median() if not r['relax_df'].empty else np.nan
    stable = r['stab_df']['stabilized'].mean() if not r['stab_df'].empty else 0
    gamma = r['gamma_df']['gamma'].mean() if not r['gamma_df'].empty else np.nan
    print(f"{label:15s}  {beta:6.3f}  {relax_med:6.3f}  {stable:6.0%}  {gamma:6.3f}")
print("=" * 70)
print(f"{'Theory':15s}  {'0.500':>6s}  {'0.667':>6s}  {'—':>7s}  {'0.500':>6s}")


Model                 β   Relax   Stable       γ
----------------------------------------------------------------------
Historic          0.552   0.066      0%   0.910
Heuristic         0.551   1.399      0%   0.893
CST               0.564   0.811      0%   0.712
LobS5             0.561   0.706      0%   0.779
CGAN              0.586   0.969      0%   0.560
Theory            0.500   0.667        —   0.500
